# 05 — MAD Assembly (Na Events)

Deze stap bouwt `MAD_shortterm` en `MAD_longterm` met weather, calendar, location **en events** in één pipeline.


In [1]:
import pandas as pd
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.parent.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.project_config import get_project_paths
from src.phase1.assembly import (
    add_coverage_flags,
    build_mad_table,
    drop_admin_columns,
    export_mad_outputs,
    load_mad_inputs,
    merge_quality_report,
    normalize_time_columns,
    prepare_calendar_for_merge,
    prepare_location_for_merge,
)

paths = get_project_paths(PROJECT_ROOT)
print(f"ROOT      : {paths.root}")
print(f"DATA_INT  : {paths.data_intermediate}")
print(f"DATA_PROC : {paths.data_processed}")



ROOT      : /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2
DATA_INT  : /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_intermediate
DATA_PROC : /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_processed


In [2]:
inputs = load_mad_inputs(paths=paths, include_events=True)

df_st = inputs["shortterm"]
df_lt = inputs["longterm"]
df_w = inputs["weather"]
df_cal = inputs["calendar"]
df_loc = inputs["location"]
df_events = inputs.get("events")

print("Geladen bestanden:")
print(f"  shortterm : {df_st.shape}")
print(f"  longterm  : {df_lt.shape}")
print(f"  weather   : {df_w.shape}")
print(f"  calendar  : {df_cal.shape}")
print(f"  location  : {df_loc.shape}")
print(f"  events    : {None if df_events is None else df_events.shape}")



Geladen bestanden:
  shortterm : (284524, 21)
  longterm  : (46643, 20)
  weather   : (52632, 19)
  calendar  : (2922, 14)
  location  : (15, 6)
  events    : (2922, 15)


In [3]:
normalize_time_columns(df_st=df_st, df_lt=df_lt, df_w=df_w)
df_cal_clean = prepare_calendar_for_merge(df_cal)
df_loc_clean = prepare_location_for_merge(df_loc)

print(f"Locatietabel na filtering: {df_loc_clean.shape}")
print(f"Unieke parkings in loc: {df_loc_clean['parking_id'].nunique()}")



Locatietabel na filtering: (11, 5)
Unieke parkings in loc: 11


In [4]:
MAD_st = build_mad_table(
    base_df=df_st,
    weather_df=df_w,
    calendar_df=df_cal_clean,
    location_df=df_loc_clean,
    events_df=df_events,
)
MAD_lt = build_mad_table(
    base_df=df_lt,
    weather_df=df_w,
    calendar_df=df_cal_clean,
    location_df=df_loc_clean,
    events_df=df_events,
)

MAD_st = add_coverage_flags(MAD_st)
MAD_lt = add_coverage_flags(MAD_lt)

MAD_st = drop_admin_columns(MAD_st)
MAD_lt = drop_admin_columns(MAD_lt)

print(f"MAD shortterm: {MAD_st.shape}")
print(f"MAD longterm : {MAD_lt.shape}")



MAD shortterm: (284524, 66)
MAD longterm : (46643, 65)


In [5]:
merge_quality_report(MAD_st, "MAD_shortterm")
merge_quality_report(MAD_lt, "MAD_longterm")

print("\nEvent-kolommen aanwezig in shortterm:")
event_cols = [
    "is_event_day","is_football_day","is_sport_day","is_festival_day",
    "is_procession_day","is_kermis_day","is_markt_day","is_carnival_day",
    "is_other_day","event_scale_max","n_concurrent_events","data_confidence",
    "event_names_combined","football_kickoff_hour"
]
print([c for c in event_cols if c in MAD_st.columns])



MERGE-KWALITEITSRAPPORT — MAD_shortterm
  rows                     : 284,524
  NaN occupancy_rate       : 0
  NaN parking_location_cat : 0
  NaN in weather core (rows): 40,455
  NaN in calendar core (rows): 5
  NaN in event core (rows)  : 0
MERGE-KWALITEITSRAPPORT — MAD_longterm
  rows                     : 46,643
  NaN occupancy_rate       : 0
  NaN parking_location_cat : 0
  NaN in weather core (rows): 0
  NaN in calendar core (rows): 0
  NaN in event core (rows)  : 0

Event-kolommen aanwezig in shortterm:
['is_event_day', 'is_football_day', 'is_sport_day', 'is_festival_day', 'is_procession_day', 'is_kermis_day', 'is_markt_day', 'is_carnival_day', 'is_other_day', 'event_scale_max', 'n_concurrent_events', 'data_confidence', 'event_names_combined', 'football_kickoff_hour']


In [6]:
outputs = export_mad_outputs(
    paths=paths,
    df_loc_clean=df_loc_clean,
    mad_st=MAD_st,
    mad_lt=MAD_lt,
)

print("Export voltooid:")
for k, v in outputs.items():
    print(f"  {k:<12}: {v}")



Export voltooid:
  location    : /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_intermediate/parking_location_clean.parquet
  mad_shortterm: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_processed/MAD_shortterm.parquet
  mad_longterm: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_processed/MAD_longterm.parquet


In [7]:
MAD_st_disk = pd.read_parquet(outputs["mad_shortterm"])
MAD_lt_disk = pd.read_parquet(outputs["mad_longterm"])

print("=== Integriteitscheck ===")
print(f"MAD_st rows: {len(MAD_st_disk):,}")
print(f"MAD_lt rows: {len(MAD_lt_disk):,}")
print(f"PK dupes MAD_st: {int(MAD_st_disk.duplicated(subset=['parking_id','rounded_hour']).sum()):,}")
print(f"PK dupes MAD_lt: {int(MAD_lt_disk.duplicated(subset=['parking_id','rounded_hour']).sum()):,}")
print(f"Eventdagen MAD_st: {int(MAD_st_disk['is_event_day'].sum()) if 'is_event_day' in MAD_st_disk.columns else 'n/a'}")

event_core_cols = [
    "is_event_day", "is_football_day", "is_sport_day", "is_festival_day",
    "is_procession_day", "is_kermis_day", "is_markt_day", "is_carnival_day",
    "is_other_day", "event_scale_max", "n_concurrent_events",
]
missing_event_cols = [c for c in event_core_cols if c not in MAD_st_disk.columns]
print(f"Ontbrekende event core kolommen in MAD_st: {missing_event_cols}")

weather_core_cols = ["temp_c", "precip_mm", "wind_speed_ms", "wind_gusts_ms", "humidity_pct", "pressure_hpa"]
present_weather_cols = [c for c in weather_core_cols if c in MAD_st_disk.columns]
if present_weather_cols:
    n_weather_nan_rows = int(MAD_st_disk[present_weather_cols].isna().any(axis=1).sum())
    print(f"Rows met missende weather core (MAD_st): {n_weather_nan_rows:,}")

calendar_core_cols = ["is_national_holiday", "is_other_holiday", "is_any_holiday", "is_school_vacation", "calendar_day_class"]
present_calendar_cols = [c for c in calendar_core_cols if c in MAD_st_disk.columns]
if present_calendar_cols:
    n_calendar_nan_rows = int(MAD_st_disk[present_calendar_cols].isna().any(axis=1).sum())
    print(f"Rows met missende calendar core (MAD_st): {n_calendar_nan_rows:,}")

if 'rounded_hour' in MAD_st_disk.columns and present_weather_cols:
    weather_start = pd.Timestamp('2020-01-01 00:00:00')
    weather_end = pd.Timestamp('2026-01-01 23:00:00')
    outside_weather = ((MAD_st_disk['rounded_hour'] < weather_start) | (MAD_st_disk['rounded_hour'] > weather_end)).sum()
    print(f"Rows buiten weather datumbereik (MAD_st): {int(outside_weather):,}")



=== Integriteitscheck ===
MAD_st rows: 284,524
MAD_lt rows: 46,643
PK dupes MAD_st: 0
PK dupes MAD_lt: 0
Eventdagen MAD_st: 70165
Ontbrekende event core kolommen in MAD_st: []
Rows met missende weather core (MAD_st): 40,455
Rows met missende calendar core (MAD_st): 5
Rows buiten weather datumbereik (MAD_st): 40,455


## INFO — MAD status na assembly (klaar voor Phase 2 EDA)

Deze sectie vat de actuele output van notebook 05 samen en geeft de minimale context die nodig is om in Phase 2 direct correct te starten.

### 1) Finale outputbestanden

| Bestand | Locatie | Vorm |
|---|---|---|
| `MAD_shortterm.parquet` | `data_processed/` | 284.524 rijen × 66 kolommen |
| `MAD_longterm.parquet` | `data_processed/` | 46.643 rijen × 65 kolommen |
| `parking_location_clean.parquet` | `data_intermediate/` | 11 rijen × 5 kolommen |

### 2) Sleutelstructuur en joins (reeds toegepast)

- Uur-join: `rounded_hour` (parking) ↔ `timestamp` (weather)
- Dag-join: `date_only` (parking) ↔ `date` (calendar/events)
- Locatie-join: `parking_id` ↔ `parking_id`
- PK-integriteit na merge: geen duplicaten op `['parking_id', 'rounded_hour']` in beide MAD-tabellen.

### 3) Dekking per MAD-tabel

| Kenmerk | `MAD_shortterm` | `MAD_longterm` |
|---|---:|---:|
| Datumbereik | 2018-12-31 23:00 → 2026-02-02 00:00 | 2024-01-01 00:00 → 2024-12-31 22:00 |
| Unieke parkings | 10 | 7 |
| Event-rijen (`is_event_day=1`) | 70.165 | 14.782 |
| Weer-core missend (rijen) | 40.455 | 0 |
| Kalender-core missend (rijen) | 5 | 0 |
| Locatiecategorie missend | 0 | 0 |

### 4) Belangrijke interpretatie voor EDA

- `MAD_shortterm` bevat rijen buiten het beschikbare weerbereik (voor 2020 en na 2026-01-01 23:00). Dit verklaart de 40.455 rijen met missende weercore.
- De 5 kalender-missings in shortterm komen overeen met `date_only=2018-12-31`, buiten de kalenderdekking in de huidige `calendar_master`.
- `MAD_longterm` is volledig binnen dekking (geen missings in weer- of kalendercore), maar is inhoudelijk beperkt tot jaar 2024.

### 5) Event-features in huidige MAD

De volgende dagflags zijn aanwezig in beide MAD-tabellen:

`is_event_day`, `is_football_day`, `is_sport_day`, `is_festival_day`, `is_procession_day`, `is_kermis_day`, `is_markt_day`, `is_carnival_day`, `is_other_day`.

Daarnaast zijn aanwezig:

`event_scale_max`, `n_concurrent_events`, `football_kickoff_hour`, `data_confidence`, `event_names_combined`.

### 6) Kwaliteitsflags en coverage-flags (relevant voor analysestrategie)

- Kwaliteitsflags uit eerdere cleaningstappen zijn behouden (o.a. frozen-sensor en weer/QC-gerelateerde flags).
- Coverage-flags in MAD:
  - `low_data_coverage`
  - `system_blackout`
  - `partial_year`

Aanbevolen voor Phase 2:

- Rapporteer EDA-resultaten zowel op volledige set als op een “strict coverage subset” (bijv. enkel rijen met complete weer+kalender) om robuustheid zichtbaar te maken.
- Behandel shortterm en longterm apart in primaire analyses; longterm is qua tijdsdekking niet rechtstreeks vergelijkbaar met shortterm.

### 7) Praktische start voor Phase 2

- Primaire EDA-dataset voor externe-factoranalyse: `MAD_shortterm.parquet`.
- Secundaire, aparte EDA-track: `MAD_longterm.parquet` (abonneeprofiel 2024).
- Bij alle inferenties over weer- of kalendereffecten: expliciet omgaan met de dekking/missingness zoals hierboven gerapporteerd.


## INFO — Volledige kolomdocumentatie MAD (shortterm + longterm)

Onderstaande tabellen documenteren **alle** kolommen in de actuele geëxporteerde MAD-bestanden.
De documentatie is bedoeld als directe referentie voor de start van Phase 2 (EDA).

### MAD_shortterm — alle kolommen

| Kolom | Type | Format | Beschrijving | Motivatie |
|---|---|---|---|---|
| `parking_id` | `str` | tekst | Unieke identifier van de parkinglocatie. | Nodig voor groepering per parking, panelanalyse en spatiale vergelijking. |
| `parking_id_hash` | `category` | categorie | Gehashte variant van `parking_id`. | Technische sleutel voor privacy-veilige koppelingen en robuuste joins. |
| `parking_type` | `str` | vrije tekst/categorie | Type parkeerproduct in de bronregistratie. | Laat segmentatie toe tussen functioneel verschillende parkeerstromen. |
| `kind` | `str` | vrije tekst/categorie | Subtype van de observatiestroom (shortterm/longterm-context uit bron). | Ondersteunt interpretatie van gebruiksregime van de observatie. |
| `year` | `int32` | integer | Jaarcomponent van de observatie. | Nodig voor seizoensanalyse, cohortvergelijking en coverage-flags. |
| `month` | `int32` | integer | Maandcomponent van de observatie. | Ondersteunt maandelijkse seizoenspatronen en interactie met weer. |
| `date_only` | `datetime64[ms]` | YYYY-MM-DD (dag) | Genormaliseerde kalenderdatum (zonder uur). | Dag-sleutel voor merge met kalender- en eventfeatures. |
| `rounded_hour` | `datetime64[ns]` | YYYY-MM-DD HH:MM:SS (uur) | Uurlijkse timestamp van de observatie. | Primaire tijdsleutel voor uur-joins en tijdreeksmodellering. |
| `hour` | `int32` | 0-23 | Uur van de dag (0–23). | Belangrijk voor intradagprofielen en piekdetectie. |
| `weekday_int` | `int32` | 0-6 (ma=0) | Weekdag als integer (0=maandag … 6=zondag). | Ondersteunt weekritmepatronen en modelmatige codering. |
| `weekday_name` | `str` | vrije tekst/categorie | Naam van de weekdag. | Leesbare categorische tijdscontext voor EDA en rapportage. |
| `day_type` | `str` | vrije tekst/categorie | Dagtype zoals aangeleverd in de bron (bv. weekdag/weekend). | Snelle operationele segmentatie van vraagprofielen. |
| `number_of_spaces_override` | `float64` | numeriek (float) | Operationele capaciteit op dat uur. | Cruciaal voor correcte bezettingsratio en detectie van capaciteitsschommelingen. |
| `number_of_occupied_spaces` | `float64` | numeriek (float) | Aantal bezette plaatsen op dat uur. | Primaire teller van parkeervraag op observatieniveau. |
| `occupancy_rate` | `float64` | ratio (0-1, mogelijk >1 bij dataprobleem) | Bezettingsgraad = bezet / operationele capaciteit. | Kernuitkomstvariabele voor voorspelling en evaluatie. |
| `flag_occ_inconsistent` | `int64` | 0/1 (binaire indicator) | Indicator voor inconsistentie tussen opgeslagen en herberekende occupancy. | Markeert potentiële meetschade of registratiefouten. |
| `flag_overcapacity` | `int64` | 0/1 (binaire indicator) | Indicator dat bezetting > operationele capaciteit. | Signaleert fysisch onmogelijke of foutief geregistreerde observaties. |
| `flag_frozen_sensor` | `int64` | 0/1 (binaire indicator) | Indicator voor lange reeksen identieke meetwaarden. | Detecteert sensorstagnatie; belangrijk voor trainingsfilters. |
| `temp_c` | `float64` | numeriek (float) | Luchttemperatuur in graden Celsius. | Exogene factor met potentiële impact op mobiliteitskeuze en parkeervraag. |
| `precip_mm` | `float64` | numeriek (float) | Neerslaghoeveelheid per uur in mm. | Sterke kandidaatverklarer voor modaliteitsverschuiving naar auto. |
| `wind_speed_ms` | `float64` | numeriek (float) | Gemiddelde windsnelheid in m/s. | Contextvariabele voor weercomfort en verplaatsingsgedrag. |
| `wind_gusts_ms` | `float64` | numeriek (float) | Windstoten in m/s. | Aanvullende weerintensiteit, relevant bij extreme omstandigheden. |
| `humidity_pct` | `float64` | numeriek (float) | Relatieve vochtigheid in %. | Aanvullende comfort- en weersituatievariabele. |
| `pressure_hpa` | `float64` | numeriek (float) | Luchtdruk in hPa. | Proxy voor weerregime (stabiel/buiig) en contextuele condities. |
| `sun_duration_min` | `float64` | numeriek (float) | Duur van zonneschijn binnen het uur (min). | Relevante indicator voor recreatief/actief verplaatsingsgedrag. |
| `shortwave_wm2` | `float64` | numeriek (float) | Globale kortgolvige instraling (W/m²). | Kwantisering van licht/straling, complementair aan zonneschijnduur. |
| `sun_intensity_wm2` | `float64` | numeriek (float) | Aanvullende zonintensiteitsmaat (W/m²). | Exogene variatie in weercondities voor fijnmazige effectanalyse. |
| `qc_temp` | `object` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus temperatuurmeting. | Maakt kwaliteitsgevoelige analyses en robustness-checks mogelijk. |
| `qc_precip` | `object` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus neerslagmeting. | Nodig om imputatie-onzekerheid expliciet te volgen. |
| `qc_wind_speed` | `object` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus windsnelheid. | Ondersteunt kwaliteitscontrole op windfeatures. |
| `qc_wind_gusts` | `object` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus windstoten. | Documenteert betrouwbaarheid van gust-waarden. |
| `qc_humidity` | `object` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus vochtigheidsmeting. | Belangrijk omdat historisch langere suspect-periodes voorkwamen. |
| `qc_pressure` | `object` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus luchtdrukmeting. | Ondersteunt filtering/gevoeligheidsanalyse op datakwaliteit. |
| `humidity_suspect` | `float64` | 0/1 (binaire indicator) | Flag voor periodes met verdachte vochtigheidskwaliteit. | Voorkomt informatieverlies terwijl onzekerheid toch gemodelleerd kan worden. |
| `sun_duration_inconsistent` | `float64` | 0/1 (binaire indicator) | Flag voor inconsistente combinatie zonduur vs instraling. | Markeert diffuse-licht randgevallen voor robuuste interpretatie. |
| `precip_imputed` | `float64` | 0/1 (binaire indicator) | Flag dat neerslagwaarde geïmputeerd werd. | Nodig voor sensitiviteitsanalyse van imputatiebeslissingen. |
| `national_holiday_name` | `str` | vrije tekst/categorie | Naam van nationale feestdag op die datum (indien van toepassing). | Semantische context voor uitzonderlijke mobiliteitspatronen. |
| `other_holiday_name` | `str` | vrije tekst/categorie | Naam van overige herdenkingsdag (indien van toepassing). | Aanvullende kalendercontext buiten officiële feestdagen. |
| `vacation_name` | `str` | vrije tekst/categorie | Naam van schoolvakantieperiode (indien van toepassing). | Ondersteunt detectie van vakantiedynamiek in vraagpatronen. |
| `vacation_type` | `str` | vrije tekst/categorie | Type schoolvakantie (krokus, paas, zomer, …). | Laat differentiële effectanalyse per vakantiecategorie toe. |
| `is_national_holiday` | `float64` | 0/1 (binaire indicator) | Binaire indicator nationale feestdag (0/1). | Direct modelbruikbare kalenderfeature. |
| `is_other_holiday` | `float64` | 0/1 (binaire indicator) | Binaire indicator overige herdenkingsdag (0/1). | Vangt niet-officiële maar potentieel relevante kalenderimpact. |
| `is_any_holiday` | `float64` | 0/1 (binaire indicator) | Binaire unie van nationale/overige feestdagen (0/1). | Compacte algemene feestdagfeature voor baseline-modellen. |
| `is_school_vacation` | `float64` | 0/1 (binaire indicator) | Binaire indicator schoolvakantie (0/1). | Essentieel voor structurele verschuivingen in verplaatsingsmotieven. |
| `calendar_day_class` | `str` | tekst | Gecombineerde dagklasse (regular/weekend/vacation/holiday). | Geaggregeerde kalenderstatus met hoge interpretabiliteit. |
| `longitude` | `float64` | numeriek (float) | Lengtegraad van de parkinglocatie. | Nodig voor ruimtelijke analyses en visualisaties. |
| `latitude` | `float64` | numeriek (float) | Breedtegraad van de parkinglocatie. | Nodig voor ruimtelijke analyses en visualisaties. |
| `total_capacity` | `float64` | numeriek (float) | Gecertificeerde totale capaciteit van de parking. | Referentiemaat voor schaalverschillen tussen parkings. |
| `parking_location_category` | `category` | categorie (centrum/vesten/rand/...) | Locatiecategorie (bv. centrum, vesten, rand). | Kernvariabele voor tier-stratified analyses volgens onderzoeksvoorstel. |
| `event_scale_max` | `int8` | ordinaal 0-3 | Maximale eventintensiteit op die dag. | Benadert potentiële druktoename door grote evenementen. |
| `n_concurrent_events` | `int8` | integer >=0 | Aantal gelijktijdige events op die dag. | Meet cumulatieve eventdruk op mobiliteit en parkeren. |
| `football_kickoff_hour` | `float64` | uur.decimaal (bv. 18.0, 20.75) | Aftrapuur voor football-events op die dag (indien aanwezig). | Voegt tijdspecifieke eventtiming toe voor intradagimpact. |
| `data_confidence` | `str` | verified | estimated | covid_modified | Confidence-label van eventdata (verified/estimated/covid_modified). | Maakt onzekerheidsbewuste interpretatie en filtering mogelijk. |
| `event_names_combined` | `str` | vrije tekst/categorie | Samengevoegde eventnamen van de dag. | Audittrail voor transparantie en handmatige validatie. |
| `is_football_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator football-eventdag (0/1). | Specifieke eventtypefeature met vaak sterke piekeffecten. |
| `is_sport_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator sporteventdag (0/1). | Meet sportgerelateerde vraagverschuiving buiten football. |
| `is_festival_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator festivaldag (0/1). | Vangt festivalgebonden vraagpieken en verblijfsdynamiek. |
| `is_procession_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator processiedag (0/1). | Meet impact van processies/optochten op parkeerdruk. |
| `is_kermis_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator kermisdag (0/1). | Belangrijk voor terugkerende lokale eventdruk. |
| `is_markt_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator marktdag/eventmarkt (0/1). | Relevante lokale vraagdriver in stedelijke kernzones. |
| `is_carnival_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator carnavaldag (0/1). | Vangt specifieke seizoensevents met atypische vraagprofielen. |
| `is_other_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator overige eventdag (0/1). | Vangt resterende eventcategorieën buiten hoofdtypes. |
| `is_event_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator dat er minstens één event is (0/1). | Algemene eventdummy voor parsimonieuze modelvarianten. |
| `low_data_coverage` | `int8` | 0/1 (binaire indicator) | Flag voor jaren met gekende lagere datadekking (0/1). | Helpt biascontrole en robuustheidsanalyse in EDA/modellen. |
| `system_blackout` | `int8` | 0/1 (binaire indicator) | Flag voor gekende storingsperiode in meetreeks (0/1). | Voorkomt misinterpretatie van structurele gaten als gedragsverandering. |
| `partial_year` | `int8` | 0/1 (binaire indicator) | Flag voor partieel jaar in de observatiehorizon (0/1). | Belangrijk bij jaar-op-jaarvergelijkingen en sample-afbakening. |

### MAD_longterm — alle kolommen

| Kolom | Type | Format | Beschrijving | Motivatie |
|---|---|---|---|---|
| `parking_id` | `str` | tekst | Unieke identifier van de parkinglocatie. | Nodig voor groepering per parking, panelanalyse en spatiale vergelijking. |
| `parking_id_hash` | `category` | categorie | Gehashte variant van `parking_id`. | Technische sleutel voor privacy-veilige koppelingen en robuuste joins. |
| `parking_type` | `str` | vrije tekst/categorie | Type parkeerproduct in de bronregistratie. | Laat segmentatie toe tussen functioneel verschillende parkeerstromen. |
| `kind` | `str` | vrije tekst/categorie | Subtype van de observatiestroom (shortterm/longterm-context uit bron). | Ondersteunt interpretatie van gebruiksregime van de observatie. |
| `year` | `int32` | integer | Jaarcomponent van de observatie. | Nodig voor seizoensanalyse, cohortvergelijking en coverage-flags. |
| `month` | `int32` | integer | Maandcomponent van de observatie. | Ondersteunt maandelijkse seizoenspatronen en interactie met weer. |
| `date_only` | `datetime64[ms]` | YYYY-MM-DD (dag) | Genormaliseerde kalenderdatum (zonder uur). | Dag-sleutel voor merge met kalender- en eventfeatures. |
| `rounded_hour` | `datetime64[ns]` | YYYY-MM-DD HH:MM:SS (uur) | Uurlijkse timestamp van de observatie. | Primaire tijdsleutel voor uur-joins en tijdreeksmodellering. |
| `hour` | `int32` | 0-23 | Uur van de dag (0–23). | Belangrijk voor intradagprofielen en piekdetectie. |
| `weekday_int` | `int32` | 0-6 (ma=0) | Weekdag als integer (0=maandag … 6=zondag). | Ondersteunt weekritmepatronen en modelmatige codering. |
| `weekday_name` | `str` | vrije tekst/categorie | Naam van de weekdag. | Leesbare categorische tijdscontext voor EDA en rapportage. |
| `day_type` | `str` | vrije tekst/categorie | Dagtype zoals aangeleverd in de bron (bv. weekdag/weekend). | Snelle operationele segmentatie van vraagprofielen. |
| `number_of_spaces_override` | `float64` | numeriek (float) | Operationele capaciteit op dat uur. | Cruciaal voor correcte bezettingsratio en detectie van capaciteitsschommelingen. |
| `number_of_occupied_spaces` | `float64` | numeriek (float) | Aantal bezette plaatsen op dat uur. | Primaire teller van parkeervraag op observatieniveau. |
| `occupancy_rate` | `float64` | ratio (0-1, mogelijk >1 bij dataprobleem) | Bezettingsgraad = bezet / operationele capaciteit. | Kernuitkomstvariabele voor voorspelling en evaluatie. |
| `flag_occ_inconsistent` | `int64` | 0/1 (binaire indicator) | Indicator voor inconsistentie tussen opgeslagen en herberekende occupancy. | Markeert potentiële meetschade of registratiefouten. |
| `flag_frozen_sensor` | `int64` | 0/1 (binaire indicator) | Indicator voor lange reeksen identieke meetwaarden. | Detecteert sensorstagnatie; belangrijk voor trainingsfilters. |
| `temp_c` | `float64` | numeriek (float) | Luchttemperatuur in graden Celsius. | Exogene factor met potentiële impact op mobiliteitskeuze en parkeervraag. |
| `precip_mm` | `float64` | numeriek (float) | Neerslaghoeveelheid per uur in mm. | Sterke kandidaatverklarer voor modaliteitsverschuiving naar auto. |
| `wind_speed_ms` | `float64` | numeriek (float) | Gemiddelde windsnelheid in m/s. | Contextvariabele voor weercomfort en verplaatsingsgedrag. |
| `wind_gusts_ms` | `float64` | numeriek (float) | Windstoten in m/s. | Aanvullende weerintensiteit, relevant bij extreme omstandigheden. |
| `humidity_pct` | `float64` | numeriek (float) | Relatieve vochtigheid in %. | Aanvullende comfort- en weersituatievariabele. |
| `pressure_hpa` | `float64` | numeriek (float) | Luchtdruk in hPa. | Proxy voor weerregime (stabiel/buiig) en contextuele condities. |
| `sun_duration_min` | `float64` | numeriek (float) | Duur van zonneschijn binnen het uur (min). | Relevante indicator voor recreatief/actief verplaatsingsgedrag. |
| `shortwave_wm2` | `float64` | numeriek (float) | Globale kortgolvige instraling (W/m²). | Kwantisering van licht/straling, complementair aan zonneschijnduur. |
| `sun_intensity_wm2` | `float64` | numeriek (float) | Aanvullende zonintensiteitsmaat (W/m²). | Exogene variatie in weercondities voor fijnmazige effectanalyse. |
| `qc_temp` | `bool` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus temperatuurmeting. | Maakt kwaliteitsgevoelige analyses en robustness-checks mogelijk. |
| `qc_precip` | `bool` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus neerslagmeting. | Nodig om imputatie-onzekerheid expliciet te volgen. |
| `qc_wind_speed` | `bool` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus windsnelheid. | Ondersteunt kwaliteitscontrole op windfeatures. |
| `qc_wind_gusts` | `bool` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus windstoten. | Documenteert betrouwbaarheid van gust-waarden. |
| `qc_humidity` | `bool` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus vochtigheidsmeting. | Belangrijk omdat historisch langere suspect-periodes voorkwamen. |
| `qc_pressure` | `bool` | True/False (of NaN bij ongedekte periode) | QC-validatiestatus luchtdrukmeting. | Ondersteunt filtering/gevoeligheidsanalyse op datakwaliteit. |
| `humidity_suspect` | `int64` | 0/1 (binaire indicator) | Flag voor periodes met verdachte vochtigheidskwaliteit. | Voorkomt informatieverlies terwijl onzekerheid toch gemodelleerd kan worden. |
| `sun_duration_inconsistent` | `int64` | 0/1 (binaire indicator) | Flag voor inconsistente combinatie zonduur vs instraling. | Markeert diffuse-licht randgevallen voor robuuste interpretatie. |
| `precip_imputed` | `int64` | 0/1 (binaire indicator) | Flag dat neerslagwaarde geïmputeerd werd. | Nodig voor sensitiviteitsanalyse van imputatiebeslissingen. |
| `national_holiday_name` | `str` | vrije tekst/categorie | Naam van nationale feestdag op die datum (indien van toepassing). | Semantische context voor uitzonderlijke mobiliteitspatronen. |
| `other_holiday_name` | `str` | vrije tekst/categorie | Naam van overige herdenkingsdag (indien van toepassing). | Aanvullende kalendercontext buiten officiële feestdagen. |
| `vacation_name` | `str` | vrije tekst/categorie | Naam van schoolvakantieperiode (indien van toepassing). | Ondersteunt detectie van vakantiedynamiek in vraagpatronen. |
| `vacation_type` | `str` | vrije tekst/categorie | Type schoolvakantie (krokus, paas, zomer, …). | Laat differentiële effectanalyse per vakantiecategorie toe. |
| `is_national_holiday` | `int8` | 0/1 (binaire indicator) | Binaire indicator nationale feestdag (0/1). | Direct modelbruikbare kalenderfeature. |
| `is_other_holiday` | `int8` | 0/1 (binaire indicator) | Binaire indicator overige herdenkingsdag (0/1). | Vangt niet-officiële maar potentieel relevante kalenderimpact. |
| `is_any_holiday` | `int8` | 0/1 (binaire indicator) | Binaire unie van nationale/overige feestdagen (0/1). | Compacte algemene feestdagfeature voor baseline-modellen. |
| `is_school_vacation` | `int8` | 0/1 (binaire indicator) | Binaire indicator schoolvakantie (0/1). | Essentieel voor structurele verschuivingen in verplaatsingsmotieven. |
| `calendar_day_class` | `str` | tekst | Gecombineerde dagklasse (regular/weekend/vacation/holiday). | Geaggregeerde kalenderstatus met hoge interpretabiliteit. |
| `longitude` | `float64` | numeriek (float) | Lengtegraad van de parkinglocatie. | Nodig voor ruimtelijke analyses en visualisaties. |
| `latitude` | `float64` | numeriek (float) | Breedtegraad van de parkinglocatie. | Nodig voor ruimtelijke analyses en visualisaties. |
| `total_capacity` | `float64` | numeriek (float) | Gecertificeerde totale capaciteit van de parking. | Referentiemaat voor schaalverschillen tussen parkings. |
| `parking_location_category` | `category` | categorie (centrum/vesten/rand/...) | Locatiecategorie (bv. centrum, vesten, rand). | Kernvariabele voor tier-stratified analyses volgens onderzoeksvoorstel. |
| `event_scale_max` | `int8` | ordinaal 0-3 | Maximale eventintensiteit op die dag. | Benadert potentiële druktoename door grote evenementen. |
| `n_concurrent_events` | `int8` | integer >=0 | Aantal gelijktijdige events op die dag. | Meet cumulatieve eventdruk op mobiliteit en parkeren. |
| `football_kickoff_hour` | `float64` | uur.decimaal (bv. 18.0, 20.75) | Aftrapuur voor football-events op die dag (indien aanwezig). | Voegt tijdspecifieke eventtiming toe voor intradagimpact. |
| `data_confidence` | `str` | verified | estimated | covid_modified | Confidence-label van eventdata (verified/estimated/covid_modified). | Maakt onzekerheidsbewuste interpretatie en filtering mogelijk. |
| `event_names_combined` | `str` | vrije tekst/categorie | Samengevoegde eventnamen van de dag. | Audittrail voor transparantie en handmatige validatie. |
| `is_football_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator football-eventdag (0/1). | Specifieke eventtypefeature met vaak sterke piekeffecten. |
| `is_sport_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator sporteventdag (0/1). | Meet sportgerelateerde vraagverschuiving buiten football. |
| `is_festival_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator festivaldag (0/1). | Vangt festivalgebonden vraagpieken en verblijfsdynamiek. |
| `is_procession_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator processiedag (0/1). | Meet impact van processies/optochten op parkeerdruk. |
| `is_kermis_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator kermisdag (0/1). | Belangrijk voor terugkerende lokale eventdruk. |
| `is_markt_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator marktdag/eventmarkt (0/1). | Relevante lokale vraagdriver in stedelijke kernzones. |
| `is_carnival_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator carnavaldag (0/1). | Vangt specifieke seizoensevents met atypische vraagprofielen. |
| `is_other_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator overige eventdag (0/1). | Vangt resterende eventcategorieën buiten hoofdtypes. |
| `is_event_day` | `int8` | 0/1 (binaire indicator) | Binaire indicator dat er minstens één event is (0/1). | Algemene eventdummy voor parsimonieuze modelvarianten. |
| `low_data_coverage` | `int8` | 0/1 (binaire indicator) | Flag voor jaren met gekende lagere datadekking (0/1). | Helpt biascontrole en robuustheidsanalyse in EDA/modellen. |
| `system_blackout` | `int8` | 0/1 (binaire indicator) | Flag voor gekende storingsperiode in meetreeks (0/1). | Voorkomt misinterpretatie van structurele gaten als gedragsverandering. |
| `partial_year` | `int8` | 0/1 (binaire indicator) | Flag voor partieel jaar in de observatiehorizon (0/1). | Belangrijk bij jaar-op-jaarvergelijkingen en sample-afbakening. |
